In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/Users/ezzeddinesayed52@gmail.com/FMCG-DE_Project/Scripts/Setups/utilities

In [0]:
print(bronze_schema,silver_schema,gold_schema)

In [0]:
dbutils.widgets.text("catalog","fmcg","Catalog")
dbutils.widgets.text("data_source","customers","Data Source")

In [0]:
data_source = dbutils.widgets.get("data_source")
catalog = dbutils.widgets.get("catalog")

base_path = (
    f"r2://sportsbar@"
    f"eefc59d684bff864c9de01ed688857a0"
    f".r2.cloudflarestorage.com/"
    f"{data_source}/*.csv"
)

print(base_path)

In [0]:
df = (
        spark.read
        .format("csv")
        .option("inferSchema", True)
        .option("header", True)
        .load(base_path)
        .withColumn("read_timestamp",F.current_timestamp())
        .select("*","_metadata.file_name","_metadata.file_size")
    )
display(df.limit(10))

In [0]:
df.printSchema()

In [0]:
(
    df.write
    .format("delta")
    .mode("overwrite")
    .option("delta.enableChangeDataFeed", "true")
    .saveAsTable(f"{catalog}.{bronze_schema}.{data_source}")
)

### Silver layer Processing

In [0]:
df_bronze = spark.sql(f"SELECT * FROM {catalog}.{bronze_schema}.{data_source}")
display(df_bronze)

In [0]:
df_bronze.groupBy("customer_id").count().where("count > 1").show()

In [0]:
print("Number of rows before droping the duplicates : ", df_bronze.count())
df_silver = df_bronze.dropDuplicates()
print("Number of rows before droping the duplicates : ", df_silver.count())


In [0]:
display(
    df_silver.filter(F.col("customer_name") != F.trim(F.col("customer_name")))
)

In [0]:
df_silver = df_silver.withColumn(
    "customer_name", F.trim(F.col("customer_name"))
)


In [0]:
df_silver.select("customer_name").distinct().show()

In [0]:
df_silver = df_silver.withColumn(
    "customer_name",
    F.when(F.col("customer_name").isNull(),None)
    .otherwise(F.initcap("customer_name"))
)

df_silver.select("customer_name").distinct().show()

In [0]:
df_silver.filter(df_silver["city"].isNull()).show()
null_customers = df_silver.filter(df_silver["city"].isNull()).select("customer_name").collect()
null_customers

In [0]:
null_customer_names = [row["customer_name"] for row in null_customers]
df_silver.filter(F.col("customer_name").isin(null_customer_names)).show()

In [0]:
customer_city_fix = {
    789403 : "Chicago",
    789603 : "Chicago",
    789420 : "Austin",
    789521 : "Seattle"
}
df_fix = spark.createDataFrame(
    [(k,v) for k,v in customer_city_fix.items()],
    ["customer_id", "fixed_city"] 
)

df_fix.show()

In [0]:
df_silver = (
    df_silver.join(df_fix, "customer_id", "left")
    .withColumn(
        "city",
        F.coalesce("city","fixed_city")
    )
    .drop("fixed_city")
    
)
display(df_silver.filter(F.col("city").isNull()))

In [0]:
df_silver.printSchema()

In [0]:
df_silver = df_silver.withColumn(
    "customer_id",
    F.col("customer_id").cast("string")
)
df_silver.printSchema()

In [0]:
df_silver.select(
    [
        F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
        for c in df_silver.columns
    ]
).show()

In [0]:
df_silver = (
    df_silver.withColumn(
    "customer",
    F.concat_ws("_", F.col("customer_name"), F.col("city"))
    )
    .withColumn(
        "market",
        F.when(F.col("city").isin(["Chicago", "Seattle"]), "North")
        .when(F.col("city").isin(["Austin"]), "South")
        .otherwise("Unknown")
    )
    .withColumn("platform",F.lit("Sports Bar"))
    .withColumn("channel",F.lit("In-Store"))
)

display(df_silver.limit(5))

In [0]:
df_silver.write\
.format("delta")\
    .option("delta.enableChangeDataFeed","true")\
    .option("mergeChema", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{silver_schema}.{data_source}")



In [0]:
%sql
SELECT *
FROM fmcg.silver.customers
limit 5

### Gold Layer Processing

In [0]:
df_gold = df_silver.select("customer_id","customer_name","customer","city","market","platform","channel")
df_gold.display(5)

In [0]:
df_gold.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed","true")\
    .option("mergeChema", "true")\
    .mode("overwrite")\
    .saveAsTable(f"{catalog}.{gold_schema}.sb_dim_{data_source}")

In [0]:
%sql
SELECT * 
FROM fmcg.gold.sb_dim_customers
LIMIT 5

In [0]:
delta_table = DeltaTable.forName(spark, "fmcg.gold.dim_customers")
df_child_customer =  spark.table("fmcg.gold.sb_dim_customers").select(
    F.col("customer_id").alias("customer_code"),
    "customer",
    "channel",
    "city",
    "market",
    "platform",
)

In [0]:
delta_table.alias("target").merge(
        source = df_child_customer.alias("source"),
        condition = "target.customer_code = source.customer_code"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()